# 超参数优化

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from torch_geometric.data import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from tqdm import tqdm
import itertools
import pandas as pd

# 加载图数据和标签
work_dir = r'D:\博士文件\博士毕业课题材料\维吾尔医药配伍机制量化分析\data'
merged_file = os.path.join(work_dir, 'all_graphs_to_be_predicted.pt')
merged_graphs = torch.load(merged_file)

# 假设每个图数据都有 y 属性表示标签，并转换为 NumPy 数组
labels = np.array([graph.y.numpy() if isinstance(graph.y, torch.Tensor) else graph.y for graph in merged_graphs])
labels = torch.tensor(labels)

# 设置随机种子
def set_seed(seed):
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

# GAT 模型定义
class GATModel(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_heads, dropout_rate=0.3):
        super(GATModel, self).__init__()
        # 第一层 GAT 注意力
        self.layer1 = GATConv(in_dim, hidden_dim, heads=num_heads, dropout=dropout_rate)
        # 第二层 GAT 注意力
        self.layer2 = GATConv(hidden_dim * num_heads, hidden_dim, heads=num_heads, dropout=dropout_rate)
        # 第三层 GAT 注意力
        self.layer3 = GATConv(hidden_dim * num_heads, hidden_dim, heads=num_heads, dropout=dropout_rate)
        # 第四层 GAT 注意力
        self.layer4 = GATConv(hidden_dim * num_heads, hidden_dim, heads=1, dropout=dropout_rate)  # 这里可以使用 `heads=1` 或者调整为更多的 heads
        # 全连接层，用于最后的分类输出
        self.fc = nn.Linear(hidden_dim, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        # 第一层 GAT
        h = self.layer1(x, edge_index)
        h = torch.relu(h)
        # 第二层 GAT
        h = self.layer2(h, edge_index)
        h = torch.relu(h)
        # 第三层 GAT
        h = self.layer3(h, edge_index)
        h = torch.relu(h)
        # 第四层 GAT
        h = self.layer4(h, edge_index)
        # 使用全局平均池化 (Global Mean Pooling)
        hg = global_mean_pool(h, batch)
        # 输出层
        out = self.fc(hg)
        return out


# 计算每个类别的 pos_weight
num_classes = labels.size(1)
pos_counts = labels.sum(dim=0)
neg_counts = labels.size(0) - pos_counts
pos_weight = neg_counts / (pos_counts + 1e-6)

# 损失函数
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# 超参数设置
param_grid = {
    'learning_rate': [0.0001,0.0003,0.0005,0.0007],  # 调整学习率范围
    'batch_size': [16,32,64,128],  # 批次大小
    #'learning_rate': [0.0003],  # 调整学习率范围
    #'batch_size': [16],  # 批次大小

    #'hidden_dim': [16,32,64,128],  # fixed for now
    #'num_heads': [2,4,6,8],  # fixed for now
    #'dropout_rate': [0.2,0.3,0.4,0.5]  # fixed for now
    'hidden_dim': [64],  # fixed for now
    'num_heads': [2],  # fixed for now
    'dropout_rate': [0.4]  # fixed for now
}

# 5折交叉验证
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(loader, model, loss_fn):
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in loader:
            batch_labels = batch.y.view(-1, num_classes)
            output = model(batch)
            loss = loss_fn(output, batch_labels)
            val_loss += loss.item()
    return val_loss / len(loader)

def create_batches(graphs, labels, batch_size):
    for i, graph in enumerate(graphs):
        graph.y = labels[i].float()
    return DataLoader(graphs, batch_size=batch_size, shuffle=True)

def train_and_evaluate(train_graphs, train_labels, val_graphs, val_labels, params, early_stopping_patience=3):
    model = GATModel(in_dim=91, hidden_dim=params['hidden_dim'], out_dim=num_classes, num_heads=params['num_heads'], dropout_rate=params['dropout_rate'])
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])

    best_val_loss = float('inf')
    patience_counter = 0
    val_losses = []

    for epoch in range(30):
        model.train()
        train_loader = create_batches(train_graphs, train_labels, batch_size=params['batch_size'])
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/50"):
            batch_labels = batch.y.view(-1, num_classes)
            optimizer.zero_grad()
            output = model(batch)
            loss = loss_fn(output, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/50, Training Loss: {avg_loss:.4f}")

        val_loader = create_batches(val_graphs, val_labels, batch_size=params['batch_size'])
        val_loss = evaluate_model(val_loader, model, loss_fn)
        print(f"Epoch {epoch + 1}/50, Validation Loss: {val_loss:.4f}")
        val_losses.append(val_loss)

        # 早停法
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print("Early stopping!")
                break

    return val_losses
# 超参数优化循环
results = []

for params in itertools.product(*param_grid.values()):
    param_dict = dict(zip(param_grid.keys(), params))
    print(f"Testing with parameters: {param_dict}")
    
    fold_losses = []
    for fold, (train_idx, val_idx) in enumerate(kf.split(merged_graphs)):
        print(f"Fold {fold + 1}/{kf.n_splits}")
        train_graphs = [merged_graphs[i] for i in train_idx]
        train_labels = labels[train_idx].float()
        val_graphs = [merged_graphs[i] for i in val_idx]
        val_labels = labels[val_idx].float()
        
        val_loss = train_and_evaluate(train_graphs, train_labels, val_graphs, val_labels, param_dict)
        fold_losses.append(val_loss)

    # 平均验证损失
    avg_fold_loss = np.mean([np.mean(losses) for losses in fold_losses])
    
    # 保存所有超参数和结果
    results.append({
        'learning_rate': param_dict['learning_rate'],
        'batch_size': param_dict['batch_size'],
        'hidden_dim': param_dict['hidden_dim'],
        'num_heads': param_dict['num_heads'],
        'dropout_rate': param_dict['dropout_rate'],
        **{f'Fold_{i+1}': np.mean(fold_losses[i]) for i in range(len(fold_losses))},
        'Avg_Val_Loss': avg_fold_loss
    })

# 保存结果到 Excel
df = pd.DataFrame(results)
output_file = os.path.join(work_dir, 'hyperparameter_optimization_results_多层注意力-lr.xlsx')
df.to_excel(output_file, index=False)
print(f"Results saved to {output_file}")

In [ ]:
# 保存结果到 Excel
df = pd.DataFrame(results, columns=['hidden_dim', 'num_heads', 'dropout_rate'] + [f'Fold_{i+1}' for i in range(5)])
output_file = os.path.join(work_dir, 'hyperparameter_optimization_results-多层注意力.xlsx')
df.to_excel(output_file, index=False)
print(f"Results saved to {output_file}")
